# Context

In `make_synthetic_questions.ipynb`, we generated synthetic questions to bootstrap evaluation of the retrieval system in our hardware store's Q&A system.

This notebook shows the first step in calculating precision and recall with different retrieval parameters. We will run more advanced experiments in future notebooks after we have these baseline scores.

## Data

Here is a brief review of the data.

In [1]:
import json
import lancedb
import pandas as pd
from typing import List, Dict
from concurrent.futures import ThreadPoolExecutor
from scoring_utils import EvalQuestion, score, score_reranked_search

pd.set_option("display.max_colwidth", 160)

db = lancedb.connect("./lancedb")
reviews_table = db.open_table("reviews")
reviews_table.to_pandas().head()

/home/msivanes/miniconda3/envs/sysrag/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


,id,product_title,product_description,review,vector
0,0,Cordless Drill,"This powerful cordless drill features an ergonomic design perfect for all-day use. With 20 torque settings and a lithium-ion battery, it offers unmatched ve...","I've been using this cordless drill for the past 6 months, and it's been a game-changer for my DIY projects. The 20 torque settings allow me to adjust the p...","[-0.0038200605, -0.009364537, -0.026297344, -0.020058312, 0.0376258, 0.0038050916, -0.000636552, 0.06768333, 0.023818497, -0.0054426882, 0.011939186, 0.0186..."
1,1,Cordless Drill,"This powerful cordless drill features an ergonomic design perfect for all-day use. With 20 torque settings and a lithium-ion battery, it offers unmatched ve...","Purchased this cordless drill a year ago and it has not disappointed. The 20 torque settings provide great control and precision, especially on delicate tas...","[-0.023377769, -0.01479075, -0.014503318, -0.029437784, 0.029629406, -0.0010449336, -0.022671165, 0.08718758, -0.0075630434, -0.0136050945, 0.041390147, 0.0..."
2,2,Cordless Drill,Our lightweight cordless drill comes equipped with a flexible LED work light to illuminate your workspace. The 18V battery provides ample power for tough ma...,"I've been using this cordless drill for the past six months on various projects around the house, and I am thoroughly impressed. The 18V battery provides in...","[-0.015202215, 0.0038308129, 0.0022391798, -0.008841734, 0.008781216, 0.008865941, -0.007449812, 0.061050933, 0.045049876, -0.01014288, 0.037472975, -0.0050..."
3,3,Cordless Drill,Our lightweight cordless drill comes equipped with a flexible LED work light to illuminate your workspace. The 18V battery provides ample power for tough ma...,"I purchased this cordless drill about a year ago for use in my small woodworking shop, and it has exceeded my expectations. The 18V lithium-ion battery prov...","[-0.025799386, -0.0032572313, -0.018315684, -0.022368867, 0.03334183, -0.0056832666, -0.02441308, 0.06597876, 0.019596254, -0.023543702, 0.05545223, 0.00619..."
4,4,Cordless Drill,"Engineered for precision, this cordless drill has a compact design that allows for maximum maneuverability in tight spaces. It includes a built-in battery i...","I've been using the Cordless Drill for about six months now, and it has exceeded my expectations. The compact design makes it easy to use in tight spaces, w...","[-0.009312706, 0.01556987, 0.0027725082, -0.0062510776, 0.012757799, 0.015630737, 0.0029307632, 0.058724828, 0.06598022, -0.004050723, 0.014145574, -0.00955..."


In [2]:
with open("synthetic_eval_dataset.json", "r") as f:
    synthetic_questions = json.load(f)
synthetic_questions[:5]
eval_questions = [EvalQuestion(**question) for question in synthetic_questions]

## Set Up Evaluation

Load the evaluation questions into a structured format.

Build a simple search function

In [3]:
eval_questions[0]

EvalQuestion(question='How strong is the power of this nail gun?', answer='The pneumatic power is very strong.', chunk_id='427', question_with_context='A user asked the following question:\nQuestion: How strong is the power of this nail gun?\nThis is about the following product:\nProduct Title: Nail Gun\nProduct Description: A pneumatic nail gun compatible with various nail sizes. The adjustable depth control ensures precise nailing.\n')

In [4]:
def run_simple_request(q: EvalQuestion, n_return_vals=5):
    results = (
        reviews_table.search(q.question_with_context).select(["id"]).limit(n_return_vals).to_list()
    )
    return [str(q.chunk_id) == str(r["id"]) for r in results]

Now do the benchmarking. For simplicity, we just compare retrieval sizes with a simple semantic search in this cell.

In [5]:
def score_simple_search(n_to_retrieve: List[int]) -> Dict[str, float]:
    # parallelize to speed this up 5-10X
    with ThreadPoolExecutor() as executor:
        hits = list(
            executor.map(lambda q: run_simple_request(q, n_to_retrieve), eval_questions)
        )
    return score(hits)

k_to_retrieve = [5, 10]
scores = pd.DataFrame([score_simple_search(n) for n in k_to_retrieve])
scores["n_retrieved"] = k_to_retrieve
scores

,precision,recall,n_retrieved
0,0.000667,0.003333,5
1,0.000778,0.007778,10


If you have Cohere set up, you can see uf a reranker improves results (we'll talk more about rerankers in the coming weeks).

In [6]:
k_to_retrieve = [5, 10]
reranked_scores = score_reranked_search(eval_questions, reviews_table, k_to_retrieve)
reranked_scores_df = pd.DataFrame([
    {"precision": scores["precision"], "recall": scores["recall"], "n_retrieved": k}
    for k, scores in reranked_scores.items()
])
print(reranked_scores_df)

   precision    recall  n_retrieved
0   0.134000  0.670000            5
1   0.096667  0.966667           10
